<a href="https://colab.research.google.com/github/HoMinhDang/llm-model-inference/blob/main/ch02/ch2_Run_LLM_With_vLLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import gc
import time

# Unload models and clean up gpu memory cache
def free_gpu(model):
  if model:
    # Removes the reference to the model's memory,
    # making it eligible for garbage collection.
    del model

  # Release any cached GPU memory that's no longer needed.
  if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

  # Trigger garbage collection to ensure memory is fully released.
  gc.collect()

free_gpu(None)


In [2]:
!pip install vllm
!pip install transformers

Run Qwen model with vLLM and track the inference time.

In [3]:
import time
from vllm import LLM, SamplingParams

model_name = "Qwen/Qwen2.5-0.5B"

# Load model with vLLM.
llm = LLM(model=model_name, dtype="float16")

# Define the prompt.
prompt = """You are an expert AI historian writing a detailed chapter for a book titled "The Evolution of Human-AI Collaboration."

Begin by summarizing the early stages of artificial intelligence in the 1950s, touching on symbolic logic and rule-based systems. Then transition into the rise of machine learning, particularly deep learning in the 2010s.

Afterward, describe how large language models like GPT transformed human-computer interaction, enabling applications in education, creative writing, customer support, and software development.

Finally, reflect on the societal and ethical implications of AI, such as misinformation, bias, and the alignment problem.

Write in a formal tone, with rich detail and examples in each era."""

# Create sampling parameters.
sampling_params = SamplingParams(temperature=0.8, top_p=0.95, max_tokens=128)

# Time the model generation.
start_time = time.time()
outputs = llm.generate([prompt], sampling_params)
end_time = time.time()

# Print the results.
for output in outputs:
  print(f"Generated text: {output}")
  print(f"Time taken: {end_time - start_time:.2f} seconds")

free_gpu(llm)


(MainProcess pid=5147) INFO 10-06 08:28:18 [api_utils.py:286] non-default args: {'dtype': 'float16', 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-0.5B'}


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

(MainProcess pid=5147) INFO 10-06 08:28:46 [model.py:699] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=5147) WARNING 10-06 08:28:46 [model.py:2442] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=5147) INFO 10-06 08:28:46 [model.py:2109] Using max model len 32768
(MainProcess pid=5147) INFO 10-06 08:28:46 [scheduler.py:308] Chunked prefill is enabled with max_num_batched_tokens=8192.


(MainProcess pid=5147) INFO 10-06 08:28:47 [kernel.py:408] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])


tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

(MainProcess pid=5147) WARNING 10-06 08:28:53 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(MainProcess pid=5147) INFO 10-06 08:31:02 [hf.py:646] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:04<00:00,  4.46s/it, est. speed input: 31.84 toks/s, output: 28.70 toks/s]

Generated text: RequestOutput(request_id=0, prompt='You are an expert AI historian writing a detailed chapter for a book titled "The Evolution of Human-AI Collaboration."\n\nBegin by summarizing the early stages of artificial intelligence in the 1950s, touching on symbolic logic and rule-based systems. Then transition into the rise of machine learning, particularly deep learning in the 2010s.\n\nAfterward, describe how large language models like GPT transformed human-computer interaction, enabling applications in education, creative writing, customer support, and software development.\n\nFinally, reflect on the societal and ethical implications of AI, such as misinformation, bias, and the alignment problem.\n\nWrite in a formal tone, with rich detail and examples in each era.', prompt_token_ids=[2610, 525, 458, 6203, 15235, 42968, 4378, 264, 11682, 12453, 369, 264, 2311, 24849, 330, 785, 37221, 315, 11097, 6691, 40, 86587, 2217, 11135, 553, 28285, 4849, 279, 4124, 17628, 315, 20443, 11

In [4]:
free_gpu(llm)

In [6]:
# Run Qwen model with standard (non-optimial) HuggingFace library and track the inference time.

In [7]:
# --- Basic Model Serving (transformers) ---
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

start_time_basic = time.time()

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B", trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B", device_map="auto", trust_remote_code=True)

# Create the pipeline.
generator = pipeline('text-generation', model=model, tokenizer=tokenizer)

outputs_basic = generator(prompt, max_length=128, temperature=0.8, top_p=0.95)
end_time_basic = time.time()

print("\n---- Basic Model Serving Results ----")
for output in outputs_basic:
    print(f"Generated text: {output['generated_text']}")
    print(f"Time taken: {end_time_basic - start_time_basic:.2f} seconds")


print(f"\nLatency difference: {(end_time_basic - start_time_basic) - (end_time - start_time):.2f} seconds")

free_gpu(generator)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_length', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=2048) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is d


---- Basic Model Serving Results ----
Generated text: You are an expert AI historian writing a detailed chapter for a book titled "The Evolution of Human-AI Collaboration."

Begin by summarizing the early stages of artificial intelligence in the 1950s, touching on symbolic logic and rule-based systems. Then transition into the rise of machine learning, particularly deep learning in the 2010s.

Afterward, describe how large language models like GPT transformed human-computer interaction, enabling applications in education, creative writing, customer support, and software development.

Finally, reflect on the societal and ethical implications of AI, such as misinformation, bias, and the alignment problem.

Write in a formal tone, with rich detail and examples in each era. a chapter that explores the evolution of human-AI collaboration, focusing on the key milestones and technological advancements that have shaped the field. Use a narrative style to engage readers and provide a comprehen

In [8]:
print(f"\nLatency difference: {(end_time_basic - start_time_basic) - (end_time - start_time):.2f} seconds")



Latency difference: 3.95 seconds
